In [ ]:
#pip install pymongo
#pip install dnspython

# 1. Implementación BD en MongoDB

In [2]:
import json
from pymongo import MongoClient
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

In [5]:
# === Configuración ===
json_file_path = "arxiv-metadata-oai-snapshot.json"
batch_size = 2000
max_workers = 8 # Ajusta según tu CPU y MongoDB

In [6]:
# === Conectar a MongoDB ===
client = MongoClient("mongodb://mongo1:30001,mongo2:30002,mongo3:30003/?replicaSet=my-replica-set&readPreference=primary&appname=MongoDB%20Compass&ssl=false")
db = client["arxiv_db"]
collection = db["articles"]

In [7]:
# === Función para insertar lote ===
def insert_batch(batch):
    try:
        collection.insert_many(batch, ordered=False)
    except Exception as e:
        print("Error al insertar batch:", e)

In [8]:
# === Contar líneas para tqdm ===
def count_lines(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        return sum(1 for _ in f)

total_lines = count_lines(json_file_path)

In [9]:
# === Lectura + carga paralela ===

with open(json_file_path, 'r', encoding='utf-8') as f, tqdm(total=total_lines, desc="Cargando") as pbar:
    batch = []
    futures = []
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        for line in f:
            try:
                record = json.loads(line)
                # 👉 Agregar campo pdf_source
                record["pdf_source"] = f"https://arxiv.org/pdf/{record['id']}"
                batch.append(record)
                if len(batch) >= batch_size:
                    future = executor.submit(insert_batch, batch)
                    futures.append(future)
                    batch = []
            except json.JSONDecodeError:
                continue
            pbar.update(1)

        # Último batch
        if batch:
            futures.append(executor.submit(insert_batch, batch))
            pbar.update(len(batch))

# === Esperar a que terminen todas las cargas ===
for future in futures:
    future.result()
print("✅ Carga paralela completa.")


Cargando:  38%|███▊      | 1039860/2754926 [00:31<03:07, 9151.44it/s] 

Error al insertar batch: mongo1:30001: [WinError 10061] No se puede establecer una conexión ya que el equipo de destino denegó expresamente dicha conexión (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms),mongo2:30002: [WinError 10061] No se puede establecer una conexión ya que el equipo de destino denegó expresamente dicha conexión (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms),mongo3:30003: [WinError 10061] No se puede establecer una conexión ya que el equipo de destino denegó expresamente dicha conexión (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms), Timeout: 30s, Topology Description: <TopologyDescription id: 684a2d4b1ce8ac439aa88383, topology_type: ReplicaSetNoPrimary, servers: [<ServerDescription ('mongo1', 30001) server_type: Unknown, rtt: None, error=AutoReconnect('mongo1:30001: [WinError 10061] No se puede establecer una conexión ya que el equipo de destino denegó expresamente dicha co

Cargando:  57%|█████▋    | 1564300/2754926 [01:08<22:54, 865.99it/s]  

Error al insertar batch: mongo1:30001: [WinError 10061] No se puede establecer una conexión ya que el equipo de destino denegó expresamente dicha conexión (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms),mongo2:30002: [WinError 10061] No se puede establecer una conexión ya que el equipo de destino denegó expresamente dicha conexión (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms),mongo3:30003: [WinError 10061] No se puede establecer una conexión ya que el equipo de destino denegó expresamente dicha conexión (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms), Timeout: 30s, Topology Description: <TopologyDescription id: 684a2d4b1ce8ac439aa88383, topology_type: ReplicaSetNoPrimary, servers: [<ServerDescription ('mongo1', 30001) server_type: Unknown, rtt: None, error=AutoReconnect('mongo1:30001: [WinError 10061] No se puede establecer una conexión ya que el equipo de destino denegó expresamente dicha co

Cargando:  68%|██████▊   | 1886283/2754926 [01:20<01:16, 11368.75it/s]

...

Error al insertar batch: not primary, full error: {'errorLabels': ['RetryableWriteError'], 'topologyVersion': {'processId': ObjectId('684a2d85f97b3564fa6c4c74'), 'counter': 9}, 'ok': 0.0, 'errmsg': 'not primary', 'code': 10107, 'codeName': 'NotWritablePrimary', '$clusterTime': {'clusterTime': Timestamp(1749691892, 1), 'signature': {'hash': b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00', 'keyId': 0}}, 'operationTime': Timestamp(1749691887, 10368)}
Error al insertar batch: not primary, full error: {'errorLabels': ['RetryableWriteError'], 'topologyVersion': {'processId': ObjectId('684a2d85f97b3564fa6c4c74'), 'counter': 10}, 'ok': 0.0, 'errmsg': 'not primary', 'code': 10107, 'codeName': 'NotWritablePrimary', '$clusterTime': {'clusterTime': Timestamp(1749691892, 2), 'signature': {'hash': b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00', 'keyId': 0}}, 'operationTime': Timestamp(1749691887, 10368)}
Error al insertar batch

Cargando: 2755852it [06:39, 4318.70it/s]                              

Error al insertar batch: not primary, full error: {'errorLabels': ['RetryableWriteError'], 'topologyVersion': {'processId': ObjectId('684a2d85f97b3564fa6c4c74'), 'counter': 20}, 'ok': 0.0, 'errmsg': 'not primary', 'code': 10107, 'codeName': 'NotWritablePrimary', '$clusterTime': {'clusterTime': Timestamp(1749692304, 2), 'signature': {'hash': b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00', 'keyId': 0}}, 'operationTime': Timestamp(1749692304, 2)}
Error al insertar batch: not primary, full error: {'errorLabels': ['RetryableWriteError'], 'topologyVersion': {'processId': ObjectId('684a2d85f97b3564fa6c4c74'), 'counter': 20}, 'ok': 0.0, 'errmsg': 'not primary', 'code': 10107, 'codeName': 'NotWritablePrimary', '$clusterTime': {'clusterTime': Timestamp(1749692304, 2), 'signature': {'hash': b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00', 'keyId': 0}}, 'operationTime': Timestamp(1749692304, 2)}
Error al insertar batch: not p

Cargando: 2755852it [15:52, 2892.50it/s]

✅ Carga paralela completa.


In [1]:

from pymongo import MongoClient

client = MongoClient("mongodb://mongo1:30001,mongo2:30002,mongo3:30003/?replicaSet=my-replica-set&readPreference=primary&appname=MongoDB%20Compass&ssl=false")
db = client["arxiv_db"]
collection = db["articles"]

# ⚠️ Eliminar todos los documentos
collection.delete_many({})
print("🗑️ Todos los documentos eliminados.")


🗑️ Todos los documentos eliminados.


# 2. Conexión a la BD y Consultas

In [ ]:
import pymongo
from pandas import DataFrame 

client = pymongo.MongoClient(
   "mongodb://mongo1:30001,mongo2:30002,mongo3:30003/?replicaSet=my-replica-set&readPreference=primary&appname=MongoDB%20Compass&ssl=false")

In [11]:
db = client["arxiv_db"]

In [12]:
collection = db['articles']

## Consultas

a. Devolver los títulos y fechas de creación de artículos publicados en el año 2025. Mostrar solo esos campos y limitar a los primeros 20 resultados.

In [13]:
docs = collection.find(
    {
        "versions": {
            "$elemMatch": {
                "version": "v1", 
                "created": {"$regex": "2025"}
            }
        }
    },
    {"title": 1, "versions.$": 1}
).limit(20)

i = 0
for doc in docs:
    i += 1
    if doc["versions"]:
        version = doc["versions"][0]
        print("Articulo", i)
        print(f"Titulo: \"{doc['title']}\"")
        print(f"Fecha: \"{version['created']}\"")
        print("-" * 50)

Articulo 1
Titulo: "Wave or Physics-Appropriate Multidimensional Upwinding Approach for
  Compressible Multiphase Flows"
Fecha: "Sun, 5 Jan 2025 01:50:58 GMT"
--------------------------------------------------
Articulo 2
Titulo: "Stationary states of forced two-phase turbulence"
Fecha: "Sun, 5 Jan 2025 01:56:30 GMT"
--------------------------------------------------
Articulo 3
Titulo: "Experiences and attitudes toward working remotely from home in a time of
  pandemic: A snapshot from a New Zealand-based online survey"
Fecha: "Sun, 5 Jan 2025 01:56:59 GMT"
--------------------------------------------------
Articulo 4
Titulo: "On the existence and regularity of weakly nonlinear stationary Boltzmann
  equations : a Fredholm alternative approach"
Fecha: "Sun, 5 Jan 2025 01:59:51 GMT"
--------------------------------------------------
Articulo 5
Titulo: "Accurate calculation of bubble and droplet properties in
  diffuse-interface two-phase simulations"
Fecha: "Sun, 5 Jan 2025 02:05:04 GMT"

b. Devolver los títulos y los autores de artículos que pertenezcan a las categorías "cs.AI" 
o "stat.ML" y que tengan al menos tres autores. Mostrar solo esos campos y limitar a los 
primeros 10 resultados. 

In [14]:
docs = collection.find(
    {
        "$and": [
            {"categories": {"$in": ["cs.AI", "stat.ML"]}},
            {"$where": "this.authors_parsed.length >= 3"}
        ]
    },
    {"title": 1, "authors": 1, "categories": 1}
).limit(10)

i = 0
for doc in docs:
    i += 1
    print("Articulo", i)
    print(f"Titulo: \"{doc['title']}\"")
    print(f"Autores: \"{doc['authors']}\"")
    print(f"Categoria: \"{doc['categories']}\"")
    print("-" * 50)


Articulo 1
Titulo: "Translating OWL and Semantic Web Rules into Prolog: Moving Toward
  Description Logic Programs"
Autores: "Ken Samuel, Leo Obrst, Suzette Stoutenberg, Karen Fox, Paul Franklin,
  Adrian Johnson, Ken Laskey, Deborah Nichols, Steve Lopez, Jason Peterson"
Categoria: "cs.AI"
--------------------------------------------------
Articulo 2
Titulo: "Decomposition During Search for Propagation-Based Constraint Solvers"
Autores: "Martin Mann and Guido Tack and Sebastian Will"
Categoria: "cs.AI"
--------------------------------------------------
Articulo 3
Titulo: "Evolving localizations in reaction-diffusion cellular automata"
Autores: "Andrew Adamatzky, Larry Bull, Pierre Collet, Emmanuel Sapin"
Categoria: "cs.AI"
--------------------------------------------------
Articulo 4
Titulo: "Classification Constrained Dimensionality Reduction"
Autores: "Raviv Raich, Jose A. Costa, Steven B. Damelin, and Alfred O. Hero III"
Categoria: "stat.ML"
-----------------------------------------

c. Devolver los títulos, las categorías y los enlaces al PDF de artículos que pertenezcan a 
la categoría "hep-ph" y tengan un DOI asignado. Mostrar solo esos campos y limitar a 15 
resultados. 

In [16]:
docs = collection.find(
    {
        "$and": [
            {"categories": "hep-ph"},
            {"doi": {"$ne": None}}
        ]
    },
    {"title": 1, "doi": 1, "categories": 1, "pdf_source": 1}
).limit(15)

i = 0
for doc in docs:
    i += 1
    print("Articulo", i)
    print(f"Titulo: \"{doc['title']}\"")
    print(f"Categoria: \"{doc['categories']}\"")
    print(f"Doi: \"{doc['doi']}\"")
    print(f"Enlace al PDF: \"{doc['pdf_source']}\"")
    print("-" * 50)


Articulo 1
Titulo: "Higgs Boson Decays into Single Photon plus Unparticle"
Categoria: "hep-ph"
Doi: "10.1103/PhysRevD.77.097701"
Enlace al PDF: "https://arxiv.org/pdf/0711.3361"
--------------------------------------------------
Articulo 2
Titulo: "Nearly tri-bimaximal mixing in the S_3 flavour symmetry"
Categoria: "hep-ph"
Doi: "10.1063/1.2965040"
Enlace al PDF: "https://arxiv.org/pdf/0712.2488"
--------------------------------------------------
Articulo 3
Titulo: "Bounds on the Simplest Little Higgs Model Mass Spectrum Through Z
  Leptonic Decay"
Categoria: "hep-ph"
Doi: "10.1103/PhysRevD.77.055001"
Enlace al PDF: "https://arxiv.org/pdf/0711.1154"
--------------------------------------------------
Articulo 4
Titulo: "Higgs Signal for h to aa at Hadron Colliders"
Categoria: "hep-ph"
Doi: "10.1088/1126-6708/2008/04/092"
Enlace al PDF: "https://arxiv.org/pdf/0712.2466"
--------------------------------------------------
Articulo 5
Titulo: "The Minimal Type-I Seesaw Model and Flavor-depen

d. Devolver los títulos, nombres de los autores y la referencia de publicación (journal-ref) 
de los artículos que tengan un DOI asignado. Mostrar solo esos campos y ordenar los 
resultados alfabéticamente por título. Limitar a los primeros 20 resultados.

In [17]:
docs = collection.find(
        {"doi": {"$ne": None}},
    {"title": 1, "journal-ref": 1, "authors": 1, "doi":1}
).limit(20).sort("title", pymongo.ASCENDING)

i = 0
for doc in docs:
    i += 1
    print("Articulo", i)
    print(f"Titulo: \"{doc['title']}\"")
    print(f"Autores: \"{doc['authors']}\"")
    print(f"Referencia de publicacion: \"{doc['journal-ref']}\"")
    print(f"Doi: \"{doc['doi']}\"")
    print("-" * 50)

Articulo 1
Titulo: "!-Graphs with Trivial Overlap are Context-Free"
Autores: "Aleks Kissinger (University of Oxford), Vladimir Zamdzhiev (University
  of Oxford)"
Referencia de publicacion: "EPTCS 181, 2015, pp. 16-31"
Doi: "10.4204/EPTCS.181.2"
--------------------------------------------------
Articulo 2
Titulo: ""$1k_F$" Singularities and Finite Density ABJM Theory at Strong Coupling"
Autores: "Oscar Henriksson and Christopher Rosen"
Referencia de publicacion: "None"
Doi: "10.1007/JHEP07(2017)009"
--------------------------------------------------
Articulo 3
Titulo: ""(Weitergeleitet von Journalistin)": The Gendered Presentation of
  Professions on Wikipedia"
Autores: "Olga Zagovora (1), Fabian Fl\"ock (1), Claudia Wagner (1 and 2) ((1)
  GESIS - Leibniz Institute for the Social Sciences, (2) University of
  Koblenz-Landau)"
Referencia de publicacion: "None"
Doi: "10.1145/3091478.3091488"
--------------------------------------------------
Articulo 4
Titulo: ""+-+" Brane Model Phenom

e. Devolver los títulos y la fecha de la primera versión (versions.created) de los artículos 
enviados entre los años 2010 y 2015. Mostrar solo esos campos y limitar a los primeros 15 
resultados.

In [18]:
docs = collection.find(
    {
        "versions": {
            "$elemMatch": {
                "version": "v1", 
                "created": {"$regex": "201[0-5]"}
            }
        }
    },
    {"title": 1, "versions.created": 1}
).limit(15)

i = 0
for doc in docs:
    i += 1
    if doc["versions"]:
        version = doc["versions"][0]
        print("Articulo", i)
        print(f"Titulo: \"{doc['title']}\"")
        print(f"Fecha: \"{version['created']}\"")
        print("-" * 50)

Articulo 1
Titulo: "Symmetry Classes"
Fecha: "Tue, 5 Jan 2010 16:05:40 GMT"
--------------------------------------------------
Articulo 2
Titulo: "Named Models in Coalgebraic Hybrid Logic"
Fecha: "Tue, 5 Jan 2010 17:25:01 GMT"
--------------------------------------------------
Articulo 3
Titulo: "VERITAS Observations of Blazars"
Fecha: "Tue, 5 Jan 2010 21:19:22 GMT"
--------------------------------------------------
Articulo 4
Titulo: "Second sound and the density response function in uniform superfluid
  atomic gases"
Fecha: "Wed, 6 Jan 2010 20:06:41 GMT"
--------------------------------------------------
Articulo 5
Titulo: "From hidden symmetry to extra dimensions: a five dimensional formulation
  of the Degenerate BESS model"
Fecha: "Fri, 15 Jan 2010 15:28:05 GMT"
--------------------------------------------------
Articulo 6
Titulo: "Stochastic Budget Optimization in Internet Advertising"
Fecha: "Fri, 15 Jan 2010 16:56:37 GMT"
--------------------------------------------------
Artic

f. Devolver los títulos, comentarios y reportes técnicos (report-no) de artículos que tengan 
comentarios definidos y no nulos. Mostrar solo esos campos, ordenando por fecha de 
actualización (update_date) en orden descendente. Limitar a 10 resultados. 

In [19]:
docs = collection.find(
        {"comments": {"$ne": None}},
    {"title": 1, "comments": 1, "report-no": 1, "update_date": 1}
).limit(10).sort("update_date", pymongo.DESCENDING)

i = 0
for doc in docs:
    i += 1
    print("Articulo", i)
    print(f"Titulo: \"{doc['title']}\"")
    print(f"Comentarios: \"{doc['comments']}\"")
    print(f"Reportes Tecnicos: \"{doc['report-no']}\"")
    print(f"Fecha de Actualizacion: \"{doc['update_date']}\"")
    print("-" * 50)

Articulo 1
Titulo: "How to avoid (apparent) signaling in Bell tests"
Comentarios: "Accepted version in Quantum"
Reportes Tecnicos: "None"
Fecha de Actualizacion: "2025-06-06"
--------------------------------------------------
Articulo 2
Titulo: "Two closed geodesics on compact bumpy Finsler manifolds"
Comentarios: "10 pages. arXiv admin note: substantial text overlap with arXiv:1803.08350; text overlap with arXiv:1504.07007 by other authors"
Reportes Tecnicos: "None"
Fecha de Actualizacion: "2025-06-06"
--------------------------------------------------
Articulo 3
Titulo: "Coupled reaction-diffusion equations on adjacent domains"
Comentarios: "54 pages, 3 figures"
Reportes Tecnicos: "None"
Fecha de Actualizacion: "2025-06-06"
--------------------------------------------------
Articulo 4
Titulo: "Linear Scaling Quantum Transport Methodologies"
Comentarios: "54 pages, 31 figures, Invited Review of Physics Reports"
Reportes Tecnicos: "None"
Fecha de Actualizacion: "2025-06-06"
-----------

# 3. Demostración de consistencia y alta disponibilidad en la base de datos

En el siguiente código se realizarán operaciones en la base de datos (insert, update y delete respectivamente), donde cada vez que se haga eso se va a apagar el nodo primario para que otro nodo reciba la operación, con esto se prueba la alta disponibilidad de la base de datos. Luego, después de realizar cada una de las operaciones se espera a que el nodo que se apagó vuelva a estar disponible y se verifica que se actualice con la información del nuevo primario, esto con el fin de probar la consistencia de los datos.

In [3]:
from pymongo import MongoClient
import subprocess
import time


hosts = ["mongo1:30001", "mongo2:30002", "mongo3:30003"]


uri = "mongodb://" + ",".join(hosts) + "/?replicaSet=my-replica-set"
client = MongoClient(uri, serverSelectionTimeoutMS=5000)
db = client["arxiv"]
collection = db["articles"]

test_doc = {
    "id": "2507.98765",
    "submitter": "Carlos Ramírez",
    "authors": "Carlos Ramírez; Elena Soto",
    "title": "Evaluating Resilience in Geo-Distributed Databases",
    "comments": "Prueba de HA y consistencia – Julio 2025",
    "journal-ref": "Reliability Journal, Vol. 5, 2025",
    "doi": "10.9999/rj.2025.005",
    "report-no": "HA-TEST-2025-02",
    "categories": "cs.DB",
    "license": "CC BY 4.0",
    "abstract": (
        "Documento sintético para verificar que, tras un failover previo al CRUD, "
        "la réplica caída se ponga al día correctamente."
    ),
    "versions": [
        {"version": "v1", "created": "Mon, 01 Jul 2025 09:00:00 GMT"}
    ],
    "update_date": "2025-07-01",
    "authors_parsed": [
        ["Ramírez", "Carlos", ""],
        ["Soto", "Elena", ""]
    ],
    "pdf_source": "https://example.org/pdf/2507.98765"
}

# ---- Funciones auxiliares ----

def get_primary():
    while True:
        pr = client.primary
        if pr:
            return f"{pr[0]}:{pr[1]}"
        time.sleep(1)

def stop_node(host_port):
    container = host_port.split(":")[0]
    subprocess.run(["docker", "stop", container], check=True)

def start_node(host_port):
    container = host_port.split(":")[0]
    subprocess.run(["docker", "start", container], check=True)
    wait_for_node(host_port)

def wait_for_node(host_port, retries=10, delay=2):
    uri = f"mongodb://{host_port}/"
    for _ in range(retries):
        try:
            tmp = MongoClient(uri, directConnection=True, serverSelectionTimeoutMS=2000)
            ism = tmp.admin.command("isMaster")
            if ism.get("ismaster") or ism.get("secondary"):
                return
        except:
            pass
        time.sleep(delay)
    raise RuntimeError(f"El nodo {host_port} no se reintegró en el set a tiempo")

def verify_docs(label, doc_id):
    print(f"\n— {label} —")
    for h in hosts:
        try:
            node = MongoClient(f"mongodb://{h}/", directConnection=True, serverSelectionTimeoutMS=2000)
            doc = node["arxiv"]["articles"].find_one({"_id": doc_id})
            if doc:
                print(f"[{h}] _id: {doc.get('_id')}, title: {doc.get('title')}, comments: {doc.get('comments')}")
            else:
                print(f"[{h}] ❌ no encontrado")
        except Exception as e:
            print(f"[{h}] 🛑 offline o error ({e})")

# ---- Secuencia de pruebas ----

old_primary = get_primary()
print(f"\n1) Primario original antes de INSERT: {old_primary}")

stop_node(old_primary)
new_primary = get_primary()
print(f"▶️ Nuevo primario para INSERT: {new_primary}")

res = collection.insert_one(test_doc)
doc_id = res.inserted_id
print(f"► Insertado _id: {doc_id}")

start_node(old_primary)
time.sleep(1)
verify_docs("Verificando INSERT en los 3 nodos", doc_id)

old_primary = get_primary()
print(f"\n2) Primario original antes de UPDATE: {old_primary}")

stop_node(old_primary)
new_primary = get_primary()
print(f"▶️ Nuevo primario para UPDATE: {new_primary}")

collection.update_one(
    {"_id": doc_id},
    {"$set": {"comments": "Actualizado tras failover – Julio 2025"}}
)
print("► UPDATE ejecutado")

start_node(old_primary)
time.sleep(1)
verify_docs("Verificando UPDATE en los 3 nodos", doc_id)

old_primary = get_primary()
print(f"\n3) Primario original antes de DELETE: {old_primary}")

stop_node(old_primary)
new_primary = get_primary()
print(f"▶️ Nuevo primario para DELETE: {new_primary}")

collection.delete_one({"_id": doc_id})
print("► DELETE ejecutado")

start_node(old_primary)
time.sleep(1)
verify_docs("Verificando DELETE en los 3 nodos", doc_id)

print("\n✅ Pruebas completadas: HA + consistencia tras failover previo.")



1) Primario original antes de INSERT: mongo1:30001
▶️ Nuevo primario para INSERT: mongo2:30002
► Insertado _id: 684a446f79d1090bec65e8bc

— Verificando INSERT en los 3 nodos —
[mongo1:30001] _id: 684a446f79d1090bec65e8bc, title: Evaluating Resilience in Geo-Distributed Databases, comments: Prueba de HA y consistencia – Julio 2025
[mongo2:30002] _id: 684a446f79d1090bec65e8bc, title: Evaluating Resilience in Geo-Distributed Databases, comments: Prueba de HA y consistencia – Julio 2025
[mongo3:30003] _id: 684a446f79d1090bec65e8bc, title: Evaluating Resilience in Geo-Distributed Databases, comments: Prueba de HA y consistencia – Julio 2025

2) Primario original antes de UPDATE: mongo2:30002
▶️ Nuevo primario para UPDATE: mongo3:30003
► UPDATE ejecutado

— Verificando UPDATE en los 3 nodos —
[mongo1:30001] _id: 684a446f79d1090bec65e8bc, title: Evaluating Resilience in Geo-Distributed Databases, comments: Actualizado tras failover – Julio 2025
[mongo2:30002] _id: 684a446f79d1090bec65e8bc, t